In [20]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, confusion_matrix

# ================== 1. LOAD ==================
df = pd.read_excel("food.xlsx")                      # same folder as the notebook
print("Rows:", len(df))

FLAGS = ["Healthy", "Good_food", "Protein_rich", "Fibre_rich", "Sugar",
         "Oil", "Ghee", "Fried", "Junk", "Refined"]
for c in FLAGS:                                       # Yes/No -> 1/0
    df[c + "_n"] = (df[c].astype(str).str.strip().str.lower() == "yes").astype(int)

# ================== 2. FILL Score AND Rating (they are empty in your file) ==================
POINTS = {"Healthy": 15, "Good_food": 15, "Protein_rich": 5, "Fibre_rich": 5,   # good things
          "Sugar": -10, "Oil": -4, "Ghee": -5, "Fried": -15, "Junk": -15, "Refined": -8}  # bad things

def make_score(flags, adjust):
    return float(np.clip(50 + sum(POINTS[c] * flags[c] for c in FLAGS) + adjust, 0, 100))

def to_rating(score):
    return "Good" if score >= 65 else ("Okay" if score >= 40 else "Not good")

df["Score"] = [make_score({c: r[c + "_n"] for c in FLAGS}, r["Adjust"]) for _, r in df.iterrows()]
df["Rating"] = df["Score"].apply(to_rating)
print(df["Rating"].value_counts(), "\n")
df.drop(columns=[c + "_n" for c in FLAGS]).to_excel("food_scored.xlsx", index=False)

# ================== 3. INPUTS (no flag columns, so it can score NEW foods) ==================
df["text"] = df["Name"].str.lower() + " | " + df["Ingredients"].str.lower()
CAT = ["Category", "Course", "Diet", "Flavor"]
X = df[["text"] + CAT]

def new_model():
    prep = ColumnTransformer([
        ("words", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), "text"),
        ("chars", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True), "text"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(C=30, max_iter=3000))])

# ================== 4. TEST (5-fold cross-validation) ==================
kf = KFold(5, shuffle=True, random_state=42)
pred_flags = {}
for f in FLAGS:
    pred_flags[f] = cross_val_predict(new_model(), X, df[f + "_n"], cv=kf)
    print(f"{f:13s} accuracy: {accuracy_score(df[f + '_n'], pred_flags[f]):.3f}")

pred_score = np.clip(50 + sum(POINTS[f] * pred_flags[f] for f in FLAGS) + df["Adjust"].values, 0, 100)
pred_rating = pd.Series(pred_score).apply(to_rating)

print("\nScore error (MAE): %.1f points" % mean_absolute_error(df["Score"], pred_score))
print("R2: %.3f" % r2_score(df["Score"], pred_score))
print("Rating accuracy: %.3f" % accuracy_score(df["Rating"], pred_rating))
labels = ["Good", "Okay", "Not good"]
print(pd.DataFrame(confusion_matrix(df["Rating"], pred_rating, labels=labels),
                   index=["real " + l for l in labels], columns=["pred " + l for l in labels]))

# ================== 5. TRAIN ON ALL DATA AND SAVE ==================
models = {f: new_model().fit(X, df[f + "_n"]) for f in FLAGS}
joblib.dump({"models": models, "flags": FLAGS, "points": POINTS}, "food_model.pkl")
print("\nSaved: food_model.pkl and food_scored.xlsx")

# ================== 6. USE IT ==================
def rate_food(name, ingredients="", category="", course="", diet="", flavor=""):
    row = pd.DataFrame([{"text": f"{name.lower()} | {ingredients.lower()}", "Category": category,
                         "Course": course, "Diet": diet, "Flavor": flavor}])
    flags = {f: int(models[f].predict(row)[0]) for f in FLAGS}
    score = make_score(flags, 0)
    return {"food": name, "score": round(score), "rating": to_rating(score),
            "flags": [f for f in FLAGS if flags[f]]}

print(rate_food("Paneer kathi roll", "refined flour, paneer, onion, oil", "Rolls, Wraps & Sandwiches", "main course", "vegetarian", "spicy"))
print(rate_food("Sprouts salad", "sprouts, onion, tomato, lemon", "Healthy Snacks, Salads & Soups", "snack", "vegetarian", "sour"))
print(rate_food("Chicken burger", "bun, chicken patty, cheese, mayonnaise", "Fast Food & Chinese", "main course", "non vegetarian", "spicy"))

Rows: 1530
Rating
Good        531
Okay        512
Not good    487
Name: count, dtype: int64 

Healthy       accuracy: 0.889
Good_food     accuracy: 0.895
Protein_rich  accuracy: 0.900
Fibre_rich    accuracy: 0.875
Sugar         accuracy: 0.951
Oil           accuracy: 0.909
Ghee          accuracy: 0.954
Fried         accuracy: 0.950
Junk          accuracy: 0.922
Refined       accuracy: 0.952

Score error (MAE): 6.7 points
R2: 0.757
Rating accuracy: 0.813
               pred Good  pred Okay  pred Not good
real Good            448         78              5
real Okay             69        411             32
real Not good         12         90            385

Saved: food_model.pkl and food_scored.xlsx
{'food': 'Paneer kathi roll', 'score': 28, 'rating': 'Not good', 'flags': ['Protein_rich', 'Oil', 'Junk', 'Refined']}
{'food': 'Sprouts salad', 'score': 90, 'rating': 'Good', 'flags': ['Healthy', 'Good_food', 'Protein_rich', 'Fibre_rich']}
{'food': 'Chicken burger', 'score': 28, 'rating': 'Not